### Блокнот чисто для тестов со Spark. **Dev-only**
Происходит преобразование сырых данных с ingestion layer в очищенные дедуплицированные строки с преобразованием типов.


In [1]:
from pyspark.sql import SparkSession

_PACKAGES = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2",
])


def createSpark():
    spark = (SparkSession.builder
             .appName("dev")
             .master("local[*]")
             .config("spark.jars.packages", _PACKAGES)
             # S3A / MinIO
             .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
             .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
             .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
             .config("spark.hadoop.fs.s3a.path.style.access", "true")
             .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
             .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
             # Iceberg extensions
             .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
             # Iceberg catalog "lake"
             .config("spark.sql.catalog.lake", "org.apache.iceberg.spark.SparkCatalog")
             .config("spark.sql.catalog.lake.type", "hadoop")
             .config("spark.sql.catalog.lake.warehouse", "s3a://iceberg-lakehouse/warehouse")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    return spark


spark = createSpark()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0f6848f2-a790-434f-b1ca-bb71a9e99fa4;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.

In [5]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, LongType, TimestampType, DoubleType

TOPIC = "tickers"
CHECKPOINT_BRONZE_PATH = f"s3a://spark-checkpoints/lakehouse/bronze/{TOPIC}_raw"
CHECKPOINT_SILVER_PATH = f"s3a://spark-checkpoints/lakehouse/silver/{TOPIC}"

# схема вложенной структуры полезной нагрузки
data_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("lastPrice", StringType(), True),
    StructField("highPrice24h", StringType(), True),
    StructField("lowPrice24h", StringType(), True),
    StructField("prevPrice24h", StringType(), True),
    StructField("volume24h", StringType(), True),
    StructField("turnover24h", StringType(), True),
    StructField("price24hPcnt", StringType(), True),
    StructField("usdIndexPrice", StringType(), True),
])

payload_schema = StructType([
    StructField("ts", LongType(), True),
    StructField("type", StringType(), True),
    StructField("cs", LongType(), True),
    StructField("topic", StringType(), True),
    StructField("data", data_schema, True),
])


PRICE_T = DecimalType(28, 12)
VOL_T   = DecimalType(38, 8)
PCT_T = DecimalType(10, 6)

source = (spark.readStream
    .format("iceberg")
    .load(f"lake.bronze.{TOPIC}_raw"))

clean = (source
    .withColumn("p", F.from_json(F.col("raw_json"), payload_schema))
    .select(
        F.col("p.data.symbol").alias("symbol"),
        F.col("p.type").alias("event_type"),
        F.col("p.cs").alias("cross_seq"),
        F.timestamp_millis(F.col("p.ts")).alias("event_ts"),
        F.col("ingestion_ts"),
        F.col("p.data.lastPrice").cast(PRICE_T).alias("last_price"),
        F.col("p.data.highPrice24h").cast(PRICE_T).alias("high_price_24h"),
        F.col("p.data.lowPrice24h").cast(PRICE_T).alias("low_price_24h"),
        F.col("p.data.prevPrice24h").cast(PRICE_T).alias("prev_price_24h"),
        F.col("p.data.usdIndexPrice").cast(PRICE_T).alias("usd_index_price"),
        F.col("p.data.volume24h").cast(VOL_T).alias("volume_24h"),
        F.col("p.data.turnover24h").cast(VOL_T).alias("turnover_24h"),
        F.col("p.data.price24hPcnt").cast(DoubleType()).alias("price_24h_pcnt"),
    )
    .where(
        F.col("symbol").isNotNull() &
        F.col("event_type").isNotNull() &
        F.col("event_ts").isNotNull() &
        F.col("ingestion_ts").isNotNull() &
        F.col("usd_index_price").isNotNull()
    )
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicates(["event_ts", "cross_seq"])
)



write = (clean.writeStream
    .format("iceberg")
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH)
    .option("fanout-enabled", "true")
    .trigger(availableNow=True)
    .outputMode("append")
    .toTable(f"lake.silver.{TOPIC}"))

write.awaitTermination()

In [4]:
spark.sql("select * from lake.silver.tickers limit 10").show()

+------+----------+---------+--------+------------+----------+--------------+-------------+--------------+---------------+----------+------------+--------------+
|symbol|event_type|cross_seq|event_ts|ingestion_ts|last_price|high_price_24h|low_price_24h|prev_price_24h|usd_index_price|volume_24h|turnover_24h|price_24h_pcnt|
+------+----------+---------+--------+------------+----------+--------------+-------------+--------------+---------------+----------+------------+--------------+
+------+----------+---------+--------+------------+----------+--------------+-------------+--------------+---------------+----------+------------+--------------+



In [6]:
write = (clean.writeStream
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="60 seconds")
    .start())
write.awaitTermination()


-------------------------------------------
Batch: 0
-------------------------------------------
+------+----------+---------+--------+------------+----------+--------------+-------------+--------------+---------------+----------+------------+--------------+
|symbol|event_type|cross_seq|event_ts|ingestion_ts|last_price|high_price_24h|low_price_24h|prev_price_24h|usd_index_price|volume_24h|turnover_24h|price_24h_pcnt|
+------+----------+---------+--------+------------+----------+--------------+-------------+--------------+---------------+----------+------------+--------------+
+------+----------+---------+--------+------------+----------+--------------+-------------+--------------+---------------+----------+------------+--------------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 